In [1]:
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from numpy import asarray
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
FUNCTIONS_HOME = '../../../functions/evaluation_functions/' #home directory of the project
REAL_DATA_HOME = '../../../data/raw/chap/' #home directory of the project
SYN_DATA_HOME  = '../../../data/processed/chap/' #home directory of the project
TRAIN_FILE = '1_Chap_Data_Real_Train.csv'
SYNTHETIC_FILE = '1_Chap_Data_Synthetic_WGANGP.csv'

In [3]:
#define directory of functions and actual directory
FUNCTIONS_DIR = FUNCTIONS_HOME + "PREPROCESSING"
ACTUAL_DIR = os.getcwd()

#change directory to functions directory
os.chdir(FUNCTIONS_DIR)

#import functions for univariate resemblance analisys
from preprocessing import DataPreProcessor

#change directory to actual directory
os.chdir(ACTUAL_DIR)

# from ydata_synthetic.synthesizers.regular import WGAN_GP
from ydata_synthetic.synthesizers.regular import RegularSynthesizer
from ydata_synthetic.synthesizers import ModelParameters, TrainParameters
print('Functions imported!!')

Functions imported!!


## Data Preprocessing

In [4]:
real_data = pd.read_csv(REAL_DATA_HOME + TRAIN_FILE)
categorical_columns = ['group']
for col in categorical_columns :
    real_data[col] = real_data[col].astype('category')
data_train = real_data
real_data

,group,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
0,Group0,2.009493,0.084640,-0.020160,0.107037,0.857051,-0.048580,0.141517,0.137274
1,Group0,1.647881,0.285048,0.750705,-0.365256,0.594206,-0.094862,0.201428,-0.240859
2,Group0,0.045838,-0.301003,-0.007641,0.189160,1.875668,-0.152879,0.003445,0.012916
3,Group0,-1.626382,0.438629,-0.352224,-0.147169,-2.445341,0.362086,-0.132354,-0.016382
4,Group0,-0.244556,-0.781058,0.191160,0.637441,-2.766078,-1.110363,0.746589,0.140332
...,...,...,...,...,...,...,...,...,...
2712,Group0,0.048656,0.340805,0.146978,-0.077990,1.468295,-0.105885,0.037067,-0.001841
2713,Group0,-3.007362,-0.110868,-0.547719,-0.061449,-3.286850,-0.079093,-0.126217,0.165070
2714,Group0,0.481546,-0.118290,-0.159216,-0.046665,0.013795,0.235801,-0.128855,0.206748
2715,Group0,0.199530,0.371273,0.268324,0.670192,0.275167,-0.382674,0.076358,0.294886


In [5]:
# data configuration
preprocessor = DataPreProcessor(data_train)
data_train = preprocessor.preprocess_train_data()
data_train

,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4,group0,group1
0,0.473990,0.137505,-0.099886,0.260214,0.215366,0.011184,0.321453,0.602461,0.206383,-0.206383
1,0.385481,0.412195,1.414785,-1.538066,0.144523,-0.051958,0.482620,-1.491322,0.206383,-0.206383
2,-0.006639,-0.391078,-0.075289,0.572899,0.489904,-0.131111,-0.049971,-0.086129,0.206383,-0.206383
3,-0.415936,0.622701,-0.752359,-0.707691,-0.674698,0.571450,-0.415285,-0.248358,0.206383,-0.206383
4,-0.077717,-1.049067,0.315336,2.279751,-0.761143,-1.437393,1.949154,0.619394,0.206383,-0.206383
...,...,...,...,...,...,...,...,...,...,...
2712,-0.005949,0.488619,0.228523,-0.444286,0.380109,-0.066997,0.040473,-0.167841,0.206383,-0.206383
2713,-0.753948,-0.130468,-1.136487,-0.381308,-0.901502,-0.030446,-0.398775,0.756374,0.206383,-0.206383
2714,0.100006,-0.140642,-0.373118,-0.325015,-0.011909,0.399161,-0.405872,0.987149,0.206383,-0.206383
2715,0.030979,0.530379,0.466956,2.404453,0.058536,-0.444616,0.146170,1.475183,0.206383,-0.206383


## Train the Model

Next, lets define the neural network for generating synthetic data. We will be using a [GAN](https://www.wikiwand.com/en/Generative_adversarial_network) network that comprises of an generator and discriminator that tries to beat each other and in the process learns the vector embedding for the data. 

The model was taken from a [Github repository](https://github.com/ydataai/gan-playground) where it is used to generate synthetic data on credit card fraud data. 

Next, lets define the training parameters for the GAN network.

In [6]:
# Training configuration
noise_dim = 128
dim = 128
batch_size = 500

log_step = 100
epochs = 300
learning_rate = 5e-4

beta_1 = 0.5
beta_2 = 0.9

models_dir = 'my_model_datasetA/'

data_dim = data_train.shape[1]
print('Shape of data: ', data_train.shape)
print('Number of columns: ', data_dim)

Shape of data:  (2717, 10)
Number of columns:  10


In [7]:
# Define model parameters using new API
model_args = ModelParameters(
    batch_size=batch_size,
    lr=learning_rate,
    betas=(beta_1, beta_2),
    noise_dim=noise_dim,
    layers_dim=dim,
    n_critic=2,  # Number of critic iterations per generator iteration
    gp_lambda=10.0  # Gradient penalty lambda for WGAN-GP
)

# Define training parameters
train_args = TrainParameters(
    epochs=epochs,
    sample_interval=log_step
)

print("Model parameters configured!")
print(f"Model: WGAN-GP")
print(f"Batch size: {batch_size}")
print(f"Learning rate: {learning_rate}")
print(f"Epochs: {epochs}")

Model parameters configured!
Model: WGAN-GP
Batch size: 500
Learning rate: 0.0005
Epochs: 300


Finally, let's run the training and see if the model is able to learn something.

In [8]:
!mkdir my_model_datasetA
!mkdir my_model_datasetA/gan
!mkdir my_model_datasetA/gan/saved

mkdir: my_model_datasetA: File exists
mkdir: my_model_datasetA/gan: File exists
mkdir: my_model_datasetA/gan/saved: File exists


In [9]:
# Training the WGAN-GP model using new API
print("Creating WGAN-GP synthesizer...")
synthesizer = RegularSynthesizer(
    modelname='wgangp',  # Use 'wgangp' for WGAN with Gradient Penalty
    model_parameters=model_args
)

# After preprocessing, all columns are numerical (categorical vars are one-hot encoded)
num_cols = list(data_train.columns)
cat_cols = []

print(f"Number of numerical columns: {len(num_cols)}")
print(f"Number of categorical columns: {len(cat_cols)}")

print("\nTraining the model...")
synthesizer.fit(
    data=data_train,
    train_arguments=train_args,
    num_cols=num_cols,
    cat_cols=cat_cols
)
print("Training completed!")

Creating WGAN-GP synthesizer...


Number of numerical columns: 10
Number of categorical columns: 0

Training the model...


  1%|▏         | 4/300 [00:00<00:39,  7.49it/s]

Epoch: 0 | disc_loss: 0.5167895555496216 | gen_loss: -0.0275475662201643
Epoch: 1 | disc_loss: 0.14125783741474152 | gen_loss: -0.013092548586428165
Epoch: 2 | disc_loss: 0.0823894664645195 | gen_loss: -0.0009077980066649616
Epoch: 3 | disc_loss: 0.06613792479038239 | gen_loss: -0.08003482967615128
Epoch: 4 | disc_loss: 0.15740710496902466 | gen_loss: 0.12569601833820343


  3%|▎         | 10/300 [00:00<00:20, 14.38it/s]

...

Epoch: 295 | disc_loss: 0.07883089035749435 | gen_loss: 0.0990782305598259
Epoch: 296 | disc_loss: 0.01657206378877163 | gen_loss: 0.08755761384963989


100%|█████████▉| 299/300 [00:31<00:00,  9.38it/s]

Epoch: 297 | disc_loss: 0.01471632719039917 | gen_loss: 0.0759827271103859
Epoch: 298 | disc_loss: 0.7154006958007812 | gen_loss: 0.1004749983549118


100%|██████████| 300/300 [00:31<00:00,  9.49it/s]

Epoch: 299 | disc_loss: 0.0619833767414093 | gen_loss: 0.10857997834682465
Training completed!


In [10]:
synthesizer.generator.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(500, 128)]              0         
                                                                 
 dense (Dense)               (500, 128)                16512     
                                                                 
 dense_1 (Dense)             (500, 256)                33024     
                                                                 
 dense_2 (Dense)             (500, 512)                131584    
                                                                 
 dense_3 (Dense)             (500, 10)                 5130      
                                                                 
Total params: 186250 (727.54 KB)
Trainable params: 186250 (727.54 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [11]:
synthesizer.critic.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(500, 10)]               0         
                                                                 
 dense_4 (Dense)             (500, 512)                5632      
                                                                 
 dropout (Dropout)           (500, 512)                0         
                                                                 
 dense_5 (Dense)             (500, 256)                131328    
                                                                 
 dropout_1 (Dropout)         (500, 256)                0         
                                                                 
 dense_6 (Dense)             (500, 128)                32896     
                                                                 
 dense_7 (Dense)             (500, 1)                  129 

## Generate data

In [12]:
size = len(data_train)
print(f"Generating {size} synthetic samples...")

# Generate synthetic data using new API
generated_samples = synthesizer.sample(n_samples=size)
generated_samples.columns = data_train.columns

print(f"Generated data shape: {generated_samples.shape}")
generated_samples

Generating 2717 synthetic samples...


Synthetic data generation: 100%|██████████| 6/6 [00:00<00:00, 150.52it/s]

Generated data shape: (3000, 10)


,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4,group0,group1
0,-2.108581,-0.725906,-1.885004,-1.880190,-1.188501,-0.242911,-2.130230,-3.205923,-2.139716,1.011624
1,-1.704843,-0.213173,-2.256733,-1.944650,-0.684592,-0.148036,-2.106706,-2.889154,-1.304963,0.045494
2,-2.125141,-0.825191,-1.681782,-1.940694,-1.313997,-0.021351,-1.962547,-3.058310,-2.623789,1.570920
3,-1.546271,-0.125571,-2.082254,-1.897275,-0.599286,0.077637,-2.084931,-2.711187,-1.304448,0.111302
4,-1.166978,0.264708,-1.967464,-1.701794,-0.117048,0.571493,-1.664608,-2.055417,-0.796451,-0.293525
...,...,...,...,...,...,...,...,...,...,...
2995,-1.724661,-0.372880,-1.804269,-1.888091,-0.843158,0.288473,-1.974008,-2.767337,-2.018383,0.955158
2996,-1.693089,-0.293187,-1.807616,-1.810659,-0.795127,0.044171,-2.089338,-2.818455,-1.614474,0.543691
2997,-2.195131,-0.757494,-1.884931,-1.939381,-1.242253,-0.304875,-2.170465,-3.270792,-2.273406,1.157572
2998,-1.936519,-0.519760,-1.956186,-1.868494,-0.994470,-0.158999,-2.152917,-3.095167,-1.836972,0.694162


## Transform and process generated data

In [13]:
import importlib
import sys
FUNCTIONS_DIR = FUNCTIONS_HOME + "PREPROCESSING"
ACTUAL_DIR = os.getcwd()

#change directory to functions directory
os.chdir(FUNCTIONS_DIR)
# 从 sys.modules 中移除旧模块
if 'preprocessing' in sys.modules:
    del sys.modules['preprocessing']

# 重新导入
from preprocessing import DataPreProcessor
os.chdir(ACTUAL_DIR)
# 重新创建 preprocessor
preprocessor = DataPreProcessor(real_data)
data_train = preprocessor.preprocess_train_data()

print("✓ 模块已重新导入")

✓ 模块已重新导入


In [14]:
synthetic_data = preprocessor.transform_data(generated_samples)
synthetic_data = synthetic_data[0:len(real_data)]
synthetic_data

,group,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
0,Group0,-8.541844,-0.545287,-0.928663,-0.455110,-4.351696,-0.234827,-0.769859,-0.550513
1,Group0,-6.892333,-0.171208,-1.117848,-0.472040,-2.482052,-0.165285,-0.761114,-0.493305
2,Group0,-8.609503,-0.617723,-0.825237,-0.471001,-4.817324,-0.072427,-0.707525,-0.523854
3,Group0,-6.244475,-0.107295,-1.029050,-0.459597,-2.165540,0.000130,-0.753020,-0.461165
4,Group0,-4.694835,0.177444,-0.970630,-0.408257,-0.376301,0.362118,-0.596771,-0.342734
...,...,...,...,...,...,...,...,...,...
2712,Group0,-5.952104,-0.110032,-0.914333,-0.425598,-1.995470,0.118183,-0.695510,-0.433784
2713,Group0,-7.202361,-0.255103,-1.023678,-0.456831,-2.909712,-0.190980,-0.777772,-0.505518
2714,Group0,-5.041869,0.095305,-0.988946,-0.426792,-0.824161,0.251926,-0.629764,-0.374986
2715,Group0,-6.180864,-0.117613,-0.936792,-0.430850,-2.062372,0.121498,-0.726012,-0.449142


In [15]:
print(real_data.dtypes, '\n', synthetic_data.dtypes)
print(real_data.shape, synthetic_data.shape)

group    category
sbp1      float64
sbp2      float64
sbp3      float64
sbp4      float64
dbp1      float64
dbp2      float64
dbp3      float64
dbp4      float64
dtype: object 
 group     object
sbp1     float64
sbp2     float64
sbp3     float64
sbp4     float64
dbp1     float64
dbp2     float64
dbp3     float64
dbp4     float64
dtype: object
(2717, 9) (2717, 9)


In [16]:
real_data.describe()

,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
count,2717.000000,2717.000000,2717.000000,2717.000000,2717.000000,2717.000000,2717.000000,2717.000000
mean,0.072962,-0.015681,0.030675,0.038696,0.057982,-0.056777,0.022021,0.028471
std,4.086347,0.729714,0.509026,0.262684,3.710970,0.733119,0.371803,0.180631
min,-18.884976,-2.732794,-2.023219,-1.289887,-17.202625,-2.580206,-1.409579,-0.862115
25%,-2.674316,-0.470025,-0.306902,-0.126936,-2.352459,-0.519699,-0.224223,-0.085791
50%,0.301318,-0.016234,0.026003,0.034561,0.244464,-0.044513,0.016090,0.032382
75%,2.979693,0.441029,0.354509,0.202381,2.653050,0.436457,0.258646,0.144028
max,11.023203,3.680704,2.233565,1.292367,12.307994,2.916243,1.550258,0.852724


In [17]:
synthetic_data.describe()

,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
count,2717.000000,2717.000000,2717.000000,2717.000000,2717.000000,2717.000000,2717.000000,2717.000000
mean,-6.545010,-0.166566,-0.943665,-0.438473,-2.382584,0.047471,-0.707441,-0.452380
std,1.270916,0.246834,0.074659,0.023709,1.340194,0.166894,0.062523,0.062874
min,-10.856389,-1.076968,-1.140287,-0.524440,-6.406606,-0.449554,-0.830323,-0.614850
25%,-7.379457,-0.322643,-0.998965,-0.453333,-3.322200,-0.072497,-0.753020,-0.500338
50%,-6.385706,-0.145956,-0.944058,-0.437415,-2.334186,0.033317,-0.721513,-0.456873
75%,-5.620048,0.009020,-0.892253,-0.422970,-1.398292,0.152602,-0.674768,-0.408747
max,-3.195981,0.513354,-0.678357,-0.355833,1.362300,0.688730,-0.385679,-0.219501


In [18]:
synthetic_data.groupby('group').count()

,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
group,,,,,,,,
Group0,2670,2670,2670,2670,2670,2670,2670,2670
Group1,47,47,47,47,47,47,47,47


In [19]:
#Save generated samples
synthetic_data.to_csv(SYN_DATA_HOME + SYNTHETIC_FILE, index=False)